In [72]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [73]:
missing_summary = df.isnull().sum()

print(missing_summary[missing_summary > 0])
print("\n")

print(df.dtypes)

lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64


lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object


In [74]:
categorical_features = df.select_dtypes(include=['object', 'category', 'str', 'string']).columns
numerical_features = df.select_dtypes(include=['number']).columns

In [75]:
df[categorical_features] = df[categorical_features].fillna('NA')
df[numerical_features] = df[numerical_features].fillna(0.0)

In [76]:
print(df.isnull().sum())

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [77]:
#answer of the 1st question:

print(df['industry'].value_counts())


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [78]:
#answer to 2nd question

correlation_matrix = df[numerical_features].corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for p1, p2 in pairs:
    print(f"{p1} and {p2}: {correlation_matrix.loc[p1, p2]:.4f}")

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [79]:
#answer to 3rd question

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# 4. Extract target variable vectors
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# 5. Remove target variable from feature DataFrames
del df_train['converted']
del df_val['converted']
del df_test['converted']


categorical_vars = ['industry', 'location', 'lead_source', 'employment_status']

for col in categorical_vars:
    score = mutual_info_score(df_train[col], y_train)
    print(f"{col}: {round(score, 2)} (raw: {score:.4f})")

industry: 0.0 (raw: 0.0020)
location: 0.0 (raw: 0.0010)
lead_source: 0.03 (raw: 0.0343)
employment_status: 0.02 (raw: 0.0190)


In [80]:
#answer to 4th question

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# 4. Train Logistic Regression model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate validation accuracy
y_pred_val = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred_val)

print(f"Validation Accuracy: {round(original_accuracy, 2)}")

Validation Accuracy: 0.65


In [81]:
#answer to 5th question

features = list(df_train.columns)
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    subset_cols = [c for c in features if c != feature]
    
    X_tr = dv.fit_transform(df_train[subset_cols].to_dict(orient='records'))
    X_v = dv.transform(df_val[subset_cols].to_dict(orient='records'))
    
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    
    acc_without = accuracy_score(y_val, m.predict(X_v))
    diff = original_accuracy - acc_without
    
    print(f"Without {feature}: Acc = {acc_without:.4f}, Difference = {diff:.4f}")

Without lead_source: Acc = 0.6420, Difference = 0.0030
Without number_of_courses_viewed: Acc = 0.6430, Difference = 0.0020
Without interaction_count: Acc = 0.6010, Difference = 0.0440


In [82]:
#answer to 6th question:

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    acc = accuracy_score(y_val, model.predict(X_val))
    print(f"C = {C:<8}: Validation Accuracy = {round(acc, 3)}")

C = 1e-06   : Validation Accuracy = 0.598
C = 1e-05   : Validation Accuracy = 0.598
C = 0.0001  : Validation Accuracy = 0.613
C = 0.001   : Validation Accuracy = 0.645
